In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import pints
import pints.io
import pints.plot
import scipy.interpolate
from collections import defaultdict
import matplotlib
import mechanistic_model

Read in data, and create a new variable (`weight2`) for weight squared.

In [ ]:
df = pd.read_csv('data/df_covariates_better_date.csv')

df['weight2'] = df['weight'] ** 2

Get the outcome variables and numbers of initial preys.

In [16]:
ns_prey_eaten = df['n_prey_eaten']
ns_prey_initial = df['n_prey_initial']

all_possible_initial_ns_prey = sorted(list(set(ns_prey_initial)))

Define two models with different configurations of the covariates.

The basic model structure is:

$$\log{a} = c_a + \beta_a x_a$$
$$\log{r} = c_r + \beta_r x_r$$

Model 1: weight covariates associated with r; all other covariates associated with a.

Model 2: a is constant; all covaraites are associated with r.

In [4]:
class FunctionalResponseRegressionModel:

    def __init__(self, X_a, X_r):
        self.X_a = X_a
        self.X_r = X_r

        self.num_a_params = X_a.shape[1]
        self.num_r_params = X_r.shape[1]

In [ ]:
# Standardize the continuous weight covariate (z-score). Using the z-score and
# its square (a) puts the linear and quadratic weight terms on a comparable,
# unit scale so a single prior SD is sensible for both, and (b) reduces the
# collinearity between them (centring largely decorrelates x and x**2).
w = df['weight']
weight_z = (w - w.mean()) / w.std()

# Model 1: all non-weight covariates load on the attack rate a; the (standardized)
# weight terms load on the digestion/handling rate r.
X_a = df[["f.acc_condH",
          "f.acc_condM",
          "f.acc_condT", 
          "f.exp_condH",
          "f.exp_condM",
          "f.exp_condT",
          "f.week6",
          "f.acc_condH:f.week6",
          "f.acc_condM:f.week6",
          "f.acc_condT:f.week6"
]].copy()
X_a.insert(0, 'Ones', 1) 

X_r = pd.DataFrame({'weight_z': weight_z, 'weight_z2': weight_z ** 2})
X_r.insert(0, 'Ones', 1)

model_1 = FunctionalResponseRegressionModel(X_a, X_r)

In [ ]:
# Model 2: a is constant (intercept only); all covariates load on r.
# Weight enters standardized (z-score and its square) alongside the 0/1 dummies.
w = df['weight']
weight_z = (w - w.mean()) / w.std()

X_a = df[[]].copy()
X_a.insert(0, 'Ones', 1) 

X_r = df[["f.acc_condH",
          "f.acc_condM", 
          "f.acc_condT",  
          "f.exp_condH",
          "f.exp_condM",
          "f.exp_condT",
          "f.week6",
          "f.acc_condH:f.week6",
          "f.acc_condM:f.week6", 
          "f.acc_condT:f.week6"]].copy()
X_r['weight_z'] = weight_z.values
X_r['weight_z2'] = (weight_z ** 2).values
X_r.insert(0, 'Ones', 1)

model_2 = FunctionalResponseRegressionModel(X_a, X_r)

Prior Predictive Checks. These are run by drawing samples from the prior, multiplying by the covariates to obtain $a$ and $r$ values, running these values through the model to obtain numbers of prey eaten, then comparing to the data.

In [ ]:
def build_gaussian_prior(
    model,
    log_a_baseline=np.log(4.0), sd_a_intercept=1.0,
    log_r_baseline=np.log(20.0), sd_r_intercept=0.5,
    slope_marginal_sd=0.75,
):
    """Weakly-informative Gaussian prior on [beta_a, beta_r], calibrated on the
    count scale rather than the coefficient scale.

    Key ideas:
    * Intercepts are recentred so the baseline predator's *simulated* prey-eaten
      distribution covers the data. The naive guess eaten ~= min(n0, r) is too
      optimistic because finite a slows capture of the last few prey, so the
      baseline that reproduces the observed ~60% median consumption (and the large
      spike of fish that eat everything) is log_r_baseline ~ log(20) with
      log_a_baseline ~ log(4) -- calibrated by simulating counts from the prior
      and matching the empirical fraction-eaten distribution, not by the eaten~=r
      heuristic (which centred the predictive too low). a stays weakly identified
      (fast rate -> flat likelihood), hence the still-modest intercept SD.
    * Slopes are regularizing and *p-aware*: with standardized covariates, the
      induced marginal SD of the linear predictor is slope_sd * sqrt(n_slopes),
      so we set per-slope SD = slope_marginal_sd / sqrt(n_slopes). This keeps the
      total covariate-driven spread of log r ~ slope_marginal_sd regardless of how
      many covariates load on it -- the old fixed variances compounded across
      covariates and pushed every draw to the 0/n0 boundaries.
    """
    n_a = model.num_a_params
    n_r = model.num_r_params
    total = n_a + n_r

    means = np.zeros(total)
    variances = np.zeros(total)

    # --- attack-rate (a) block: intercept then slopes ---
    means[0] = log_a_baseline
    variances[0] = sd_a_intercept ** 2
    n_a_slopes = n_a - 1
    if n_a_slopes > 0:
        variances[1:n_a] = (slope_marginal_sd / np.sqrt(n_a_slopes)) ** 2

    # --- digestion-rate (r) block: intercept then slopes ---
    means[n_a] = log_r_baseline
    variances[n_a] = sd_r_intercept ** 2
    n_r_slopes = n_r - 1
    if n_r_slopes > 0:
        variances[n_a + 1:] = (slope_marginal_sd / np.sqrt(n_r_slopes)) ** 2

    return pints.MultivariateGaussianLogPrior(means, np.diag(variances))


prior_model_1 = build_gaussian_prior(model_1)

In [ ]:
def prior_predictive(prior, num_prior_samples, model):

    prior_a_values = []
    prior_r_values = []
    prior_frac_eaten = []  # induced E[fraction of prey eaten] per (draw, fish)
    prior_dists_over_num_eaten = defaultdict(list)
    data_dists_over_num_eaten = defaultdict(list)

    prior_samples = prior.sample(num_prior_samples)

    X_a = model.X_a
    X_r = model.X_r

    for x in prior_samples:

        for n_prey_eaten, n_prey_initial, x_a, x_r in zip(
            ns_prey_eaten, ns_prey_initial, X_a.to_numpy(), X_r.to_numpy()
        ):
        
            beta_a = x[:model.num_a_params]
            beta_r = x[model.num_a_params:]

            a = mechanistic_model.a_parameter_creator(beta_a, x_a)
            r = mechanistic_model.a_parameter_creator(beta_r, x_r)

            prior_a_values.append(a)
            prior_r_values.append(r)
        
            p_eaten = mechanistic_model.solve_system_simple_h_simplex(n_prey_initial, a, r)
            p_eaten = np.asarray(p_eaten)

            prior_dists_over_num_eaten[n_prey_initial].append(p_eaten)

            # Induced expected number eaten under this prior draw, as a fraction of
            # the initial prey. This is the count-scale quantity we actually care
            # about: a good prior should place most of this mass in the interior
            # (0, 1) rather than piling it at 0 or 1 (the saturation boundaries).
            expected_eaten = np.dot(np.arange(n_prey_initial + 1), p_eaten)
            prior_frac_eaten.append(expected_eaten / n_prey_initial)

            # For the data, create a vector of 0 in all possibilities, except for a 
            # 1 for the actual number of preys eaten.
            data = [0] * (n_prey_initial + 1)
            data[n_prey_eaten] = 1
            data_dists_over_num_eaten[n_prey_initial].append(np.asarray(data))

    return (prior_a_values, prior_r_values, prior_frac_eaten,
            prior_dists_over_num_eaten, data_dists_over_num_eaten)

In [ ]:
# Run for Model 1

(prior_a_values, prior_r_values, prior_frac_eaten,
 prior_dists_over_num_eaten, data_dists_over_num_eaten) = prior_predictive(
    prior_model_1,
    100,
    model_1,
)

fig = plt.figure(figsize=(8, 6.5))

for i, n in enumerate(all_possible_initial_ns_prey):
    ax = fig.add_subplot(3, 3, i+1)
    
    m = prior_dists_over_num_eaten[n]
    d = data_dists_over_num_eaten[n]

    x = np.arange(n+1)
    l1, = ax.plot(x, np.mean(m, axis=0), label='Inferred')
    p1 = ax.fill_between(x, np.percentile(m, 2.5, axis=0), np.percentile(m, 97.5, axis=0), alpha=0.25)

    l2, = ax.plot(x, np.mean(d, axis=0), color='k', label='Data')

    ax.legend(((l1, p1), l2), ['Prior', 'Data'])

    ax.set_xlabel('Num Prey eaten')
    ax.set_ylabel('Probability')

fig.set_tight_layout(True)
plt.show()

fig = plt.figure(figsize=(9, 3))
ax = fig.add_subplot(1, 3, 1)
ax.hist(np.log(prior_a_values))
ax.set_xlabel(r'$\log(a)$')
ax.set_ylabel('Count')

ax = fig.add_subplot(1, 3, 2)
ax.hist(np.log(prior_r_values))
ax.set_xlabel(r'$\log(r)$')

# Count-scale check: induced prior on the expected fraction of prey eaten.
# We want mass spread across the interior (not piled at 0 or 1), bracketing the
# observed mean fraction eaten (black line).
ax = fig.add_subplot(1, 3, 3)
ax.hist(prior_frac_eaten, bins=20, range=(0, 1))
ax.axvline((ns_prey_eaten / ns_prey_initial).mean(), color='k', label='Data mean')
ax.set_xlabel('E[fraction eaten]')
ax.legend()

fig.set_tight_layout(True)
plt.show()


In [ ]:
# Model 2 puts every covariate on r, so the p-aware slope scaling in
# build_gaussian_prior does the important work of stopping the ~12 covariate
# effects from compounding into an enormous marginal variance on log r.
prior_model_2 = build_gaussian_prior(model_2)

In [ ]:
# Run for Model 2

(prior_a_values, prior_r_values, prior_frac_eaten,
 prior_dists_over_num_eaten, data_dists_over_num_eaten) = prior_predictive(
    prior_model_2,
    100,
    model_2,
)

fig = plt.figure(figsize=(8, 6.5))

for i, n in enumerate(all_possible_initial_ns_prey):
    ax = fig.add_subplot(3, 3, i+1)
    
    m = prior_dists_over_num_eaten[n]
    d = data_dists_over_num_eaten[n]

    x = np.arange(n+1)
    l1, = ax.plot(x, np.mean(m, axis=0), label='Inferred')
    p1 = ax.fill_between(x, np.percentile(m, 2.5, axis=0), np.percentile(m, 97.5, axis=0), alpha=0.25)

    l2, = ax.plot(x, np.mean(d, axis=0), color='k', label='Data')

    ax.legend(((l1, p1), l2), ['Prior', 'Data'])

    ax.set_xlabel('Num Prey eaten')
    ax.set_ylabel('Probability')

fig.set_tight_layout(True)
plt.show()

fig = plt.figure(figsize=(9, 3))
ax = fig.add_subplot(1, 3, 1)
ax.hist(np.log(prior_a_values))
ax.set_xlabel(r'$\log(a)$')
ax.set_ylabel('Count')

ax = fig.add_subplot(1, 3, 2)
ax.hist(np.log(prior_r_values))
ax.set_xlabel(r'$\log(r)$')

# Count-scale check: induced prior on the expected fraction of prey eaten.
ax = fig.add_subplot(1, 3, 3)
ax.hist(prior_frac_eaten, bins=20, range=(0, 1))
ax.axvline((ns_prey_eaten / ns_prey_initial).mean(), color='k', label='Data mean')
ax.set_xlabel('E[fraction eaten]')
ax.legend()

fig.set_tight_layout(True)
plt.show()


Run Inference

In [ ]:
class Likelihood(pints.LogPDF):
    def __init__(self, ns_prey_eaten, ns_prey_initial, model, parallel=True, n_jobs=-1):
        super().__init__()
        self.ns_prey_eaten = ns_prey_eaten
        self.ns_prey_initial = ns_prey_initial

        self.X_a = model.X_a
        self.X_r = model.X_r

        self.num_a_params = model.num_a_params
        self.num_r_params = model.num_r_params

        # Parallelise the 668 independent per-fish solves across processes. The
        # parallel path is bit-for-bit identical to the serial one (same solves,
        # same left-to-right sum); it only distributes the work. ~8x on this box.
        self.parallel = parallel
        self.n_jobs = n_jobs

    def __call__(self, x):
        beta_a = x[:self.num_a_params]
        beta_r = x[self.num_a_params:]

        if self.parallel:
            ll, _ = mechanistic_model.log_likelihood_all_fish_parallel(
                self.ns_prey_eaten, self.ns_prey_initial, beta_a, beta_r,
                self.X_a, self.X_r, n_jobs=self.n_jobs)
        else:
            ll, _ = mechanistic_model.log_likelihood_all_fish(
                self.ns_prey_eaten, self.ns_prey_initial, beta_a, beta_r,
                self.X_a, self.X_r)
        return ll
    
    def n_parameters(self):
        return self.num_a_params + self.num_r_params

In [ ]:
ll_1 = Likelihood(ns_prey_eaten, ns_prey_initial, model_1)

posterior = pints.LogPosterior(ll_1, prior_model_1)

best_opt_result = -np.inf
best_opt_x = -1
for i in range(10):
    opt = pints.OptimisationController(
        posterior,
        prior_model_1.sample(),
        method=pints.CMAES,
    )
    opt.set_max_iterations(400)
    x, err = opt.run()
    if err > best_opt_result:
        best_opt_x = np.copy(x)
        best_opt_result = err
    
x = best_opt_x

# Disperse the chain starting points instead of starting all three at the single
# optimum. Starting every chain at the same point makes R-hat/ESS look healthy
# even when the chains never explore, and hides multimodality / the weakly
# identified a-plateau. Jittering by ~0.3 on the log scale gives an honest
# convergence check while staying in a sensible region.
np.random.seed(0)
n_params = posterior.n_parameters()
inits = [best_opt_x + np.random.normal(0, 0.3, n_params) for _ in range(3)]

mcmc = pints.MCMCController(
    posterior,
    3,
    inits,
    method=pints.HaarioBardenetACMC,
)

mcmc.set_max_iterations(50000)

mcmc.set_chain_filename('outputs/model_1_mcmc.csv')

chains_opt = mcmc.run()

In [ ]:
ll_2 = Likelihood(ns_prey_eaten, ns_prey_initial, model_2)

posterior = pints.LogPosterior(ll_2, prior_model_2)

best_opt_result = -np.inf
best_opt_x = -1
for i in range(10):
    opt = pints.OptimisationController(
        posterior,
        prior_model_2.sample(),  # was prior_model_1 (wrong prior for Model 2)
        method=pints.CMAES,
    )
    opt.set_max_iterations(400)
    x, err = opt.run()
    if err > best_opt_result:
        best_opt_x = np.copy(x)
        best_opt_result = err

x = best_opt_x

# Disperse the chain starting points (see Model 1 cell for rationale).
np.random.seed(0)
n_params = posterior.n_parameters()
inits = [best_opt_x + np.random.normal(0, 0.3, n_params) for _ in range(3)]

mcmc = pints.MCMCController(
    posterior,
    3,
    inits,
    method=pints.HaarioBardenetACMC,
)

mcmc.set_max_iterations(50000)

mcmc.set_chain_filename('outputs/model_2_mcmc.csv')

chains_opt = mcmc.run()